
# Генераторы в Python: учебный ноутбук (очень подробно)

Этот ноутбук — пошаговое руководство, чтобы **понять “на ощущениях”**, что такое генераторы, зачем они нужны и как ими пользоваться.

**Что ты получишь в конце:**
- чёткое понимание разницы: *итерируемое* → *итератор* → *генератор*;
- уверенность в `yield`, `yield from`, генераторных выражениях;
- практические паттерны: ленивые пайплайны, большие файлы, бесконечные последовательности;
- “продвинутый уровень”: `send()`, `throw()`, `close()` и аккуратное завершение.

> Совет: выполняй ячейки по порядку и иногда **меняй код** (это лучший способ “прочувствовать”).



## 0. Быстрый ориентир

**Слова, которые будут встречаться:**

- **iterable (итерируемое)** — объект, по которому можно пройтись в цикле `for`.
  - Примеры: `list`, `tuple`, `str`, `dict`, `set`, `range`, файл, и т.д.
- **iterator (итератор)** — объект, который *выдаёт элементы по одному* через `next()` и помнит своё состояние.
- **generator (генератор)** — *удобный способ создать итератор* с помощью функции с `yield` (или генераторного выражения).

Главная идея: **генераторы делают вычисления “ленивыми”** — значения создаются **по требованию**, а не заранее.



## 1. Итерация в Python под капотом

Цикл

```python
for x in something:
    ...
```

примерно делает:

```python
it = iter(something)     # получить итератор
while True:
    try:
        x = next(it)     # взять следующий элемент
    except StopIteration:
        break
    ...
```

Давай посмотрим это живьём.


In [2]:

data = [10, 20, 30]
it = iter(data)

print(it)                # это итератор
print(next(it))
print(next(it))
print(next(it))

# Следующий next(it) выбросит StopIteration


10
20
30


In [5]:

data = [10, 20, 30]
it = iter(data)


for i in it:
    print(i)

10
20
30


In [6]:

try:
    print(next(it))
except StopIteration as e:
    print("StopIteration пойман — итератор закончился.")


StopIteration пойман — итератор закончился.



### Важный “факт ощущения”
Итератор — это **одноразовый** объект: как только элементы закончились, он “пустой”.


In [7]:

it2 = iter([1, 2, 3])
print(list(it2))  # "съели" итератор
print(list(it2))  # второй раз уже нечего


[1, 2, 3]
[]



## 2. Где здесь генераторы?

Генератор — это итератор, который обычно создают так:

- **генераторная функция**: содержит `yield`
- **генераторное выражение**: `(x*x for x in ...)`

Начнём с сравнения “список сразу” vs “ленивый поток”.


In [8]:

def squares_list(n):
    # создаёт сразу ВСЕ значения
    return [i*i for i in range(n)]

def squares_gen(n):
    # создаёт значения ПО ОДНОМУ
    for i in range(n):
        yield i*i

print(squares_list(5))
print(squares_gen(5))  # это не список!


[0, 1, 4, 9, 16]
<generator object squares_gen at 0x71d2263a0930>


In [9]:

g = squares_gen(5)
print("type:", type(g))
print(next(g))
print(next(g))
print(list(g))  # доедаем оставшееся


type: <class 'generator'>
0
1
[4, 9, 16]



### Что делает `yield`?

`yield` **останавливает выполнение функции**, *возвращает значение наружу* и **запоминает место**, где остановились.

Когда ты вызываешь `next()` снова — выполнение продолжается **сразу после `yield`**.


In [10]:

def demo():
    print("A: старт")
    yield 1
    print("B: после первого yield")
    yield 2
    print("C: после второго yield")
    yield 3
    print("D: конец")

g = demo()
print("Создали генератор, но он ещё ничего не печатал.")

print("next #1 ->", next(g))
print("next #2 ->", next(g))
print("next #3 ->", next(g))

try:
    print("next #4 ->", next(g))
except StopIteration:
    print("Генератор завершился (StopIteration).")


Создали генератор, но он ещё ничего не печатал.
A: старт
next #1 -> 1
B: после первого yield
next #2 -> 2
C: после второго yield
next #3 -> 3
D: конец
Генератор завершился (StopIteration).



## 3. Генератор = “состояние + шаг выдачи”

Обычно генераторы используют, когда:
- данных много (не хочется держать всё в памяти),
- вычисления дорогие (не хочется считать лишнее),
- хочется строить “поток” преобразований (пайплайн).

Давай почувствуем экономию памяти.


In [14]:

import sys

n = 1_000_00  # 100k, можно увеличить

lst = [i for i in range(n)]
gen = (i for i in range(n))

print(type(gen))

print("Размер списка (байт):", sys.getsizeof(lst))
print("Размер генератора (байт):", sys.getsizeof(gen))

# Важно: getsizeof не считает память элементов списка — но уже видно,
# что у генератора "контейнер" крошечный, а список ощутимее.


<class 'generator'>
Размер списка (байт): 800984
Размер генератора (байт): 192



## 4. Простой практический паттерн: “поток преобразований”

Сделаем пайплайн:
1) взять числа
2) отфильтровать
3) преобразовать
4) взять первые N

Важно: **ни один шаг не будет хранить всё целиком**.


In [15]:

def numbers():
    i = 0
    while True:
        yield i
        i += 1

def only_even(iterable):
    for x in iterable:
        if x % 2 == 0:
            yield x

def square(iterable):
    for x in iterable:
        yield x * x

def take(iterable, n):
    it = iter(iterable)
    for _ in range(n):
        yield next(it)

pipeline = take(square(only_even(numbers())), 10)
print(list(pipeline))


[0, 4, 16, 36, 64, 100, 144, 196, 256, 324]



### Обсуждение
- `numbers()` — **бесконечный генератор**
- `only_even()` и `square()` — генераторы, которые “оборачивают” другой поток
- `take()` — ограничивает поток

Такую композицию удобно читать, потому что каждый шаг прост.



## 5. Генераторные выражения

Это “компактная форма” генератора:

- список: `[expr for x in it if cond]`
- генератор: `(expr for x in it if cond)`

Разница ощущается по памяти и ленивости.


In [16]:

nums = range(10)

lst = [x*x for x in nums if x % 2 == 0]
gen = (x*x for x in nums if x % 2 == 0)

print("list:", lst)
print("gen :", gen)
print("gen -> list:", list(gen))


list: [0, 4, 16, 36, 64]
gen : <generator object <genexpr> at 0x71d2263a1150>
gen -> list: [0, 4, 16, 36, 64]



## 6. `return` в генераторе и значение StopIteration

В генераторной функции можно написать `return value`.
Тогда генератор завершится, а `value` окажется внутри `StopIteration.value`.

На практике это редко нужно, но полезно понимать.


In [21]:

def gen_with_return():
    yield "шаг 1"
    yield "шаг 2"
    return "готово!"

g = gen_with_return()
print(next(g))
print(next(g))
try:
    next(g)
except StopIteration as e:
    print("StopIteration.value =", e.value)


шаг 1
шаг 2
StopIteration.value = готово!



## 7. `yield from`: делегирование генератора

`yield from other_iterable` делает две вещи:
1) выдаёт все элементы `other_iterable`
2) аккуратно прокидывает `send/throw/close` (это важно для продвинутых случаев)

Сравни “вручную” и через `yield from`.


In [28]:

def chain_manual(a, b):
    for x in a:
        yield x
    for x in b:
        yield x

def chain_yield_from(a, b):
    yield from a
    yield from b


gen = chain_manual([1,2], "ab")
print(next(gen))
print(next(gen))
print(next(gen))
print(next(gen))


print(list(chain_manual([1,2], "ab")))
print(list(chain_yield_from([1,2], "ab")))


1
2
a
b
[1, 2, 'a', 'b']
[1, 2, 'a', 'b']



## 8. Реальная задача: читать большой файл построчно

Файл в Python сам по себе — итерируемый объект: строки читаются **лениво**.

Но часто хочется:
- пропускать пустые строки,
- убирать пробелы,
- брать только первые N строк и т.д.

Сделаем “streaming” обработку.


In [29]:

from pathlib import Path

p = Path("demo_big_text.txt")
p.write_text("\n".join([
    "  first  ",
    "",
    "second",
    "   third",
    "",
    "fourth"
]), encoding="utf-8")

def clean_lines(path):
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                yield line

print(list(clean_lines(p)))


['first', 'second', 'third', 'fourth']



### Важное замечание
Генератор `clean_lines` открывает файл в `with` и **закрывает его сам** после завершения итерации.

Но если ты **не доитерируешь** генератор, файл может оставаться открыт до сборки мусора.
Ниже покажем, как гарантировать закрытие через `try/finally` и `close()`.



## 9. Аккуратное завершение: `close()` и `finally`

Когда ты вызываешь `g.close()`, в генератор бросается `GeneratorExit`.
Если внутри есть `finally`, он выполнится.

Это важно для ресурсов: файлы, соединения, блокировки.


In [30]:

def resource_demo():
    print("Открыли ресурс")
    try:
        yield 1
        yield 2
        yield 3
    finally:
        print("Закрыли ресурс (finally)")

g = resource_demo()
print(next(g))
g.close()  # досрочно закрыли


Открыли ресурс
1
Закрыли ресурс (finally)



## 10. Продвинутый режим: `send()` — генератор как корутина

Обычный `next(g)` эквивалентен `g.send(None)`.

Если внутри генератора есть выражение вида:

```python
x = yield something
```

то значение, отправленное через `.send(value)`, попадёт в переменную `x`.

Это позволяет делать генераторы “приёмниками”.


In [1]:

def accumulator():
    total = 0
    while True:
        x = yield total  # выдаём текущее, а затем получаем новое через send()
        if x is None:
            continue
        total += x

acc = accumulator()

# "запуск" (prime): первый next/ send(None) до первого yield
print("prime ->", next(acc))

print("send 10 ->", acc.send(10))
print("send 5  ->", acc.send(5))
print("send 1  ->", acc.send(1))


prime -> 0
send 10 -> 10
send 5  -> 15
send 1  -> 16



### Почему нужен “prime”?
Пока генератор не дошёл до первого `yield`, ему “некуда” принимать значение.

Поэтому обычно делают:
- `next(gen)` один раз,
- или пишут обёртку-декоратор для автозапуска (ниже).


In [ ]:

from functools import wraps

def coroutine(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        g = func(*args, **kwargs)
        next(g)  # auto-prime
        return g
    return wrapper

@coroutine
def moving_average():
    values = []
    while True:
        x = yield (sum(values) / len(values)) if values else None
        values.append(x)

ma = moving_average()
print(ma.send(10))
print(ma.send(20))
print(ma.send(30))



## 11. `throw()` — бросить исключение внутрь генератора

Иногда нужно “сигнализировать” генератору из внешнего кода:
- “остановись”
- “перейди в режим X”
- “произошла ошибка”

Тогда используют `g.throw(SomeError(...))`.


In [ ]:

class StopNow(Exception):
    pass

def controlled_counter():
    i = 0
    try:
        while True:
            try:
                yield i
                i += 1
            except ValueError:
                # пример обработки "команды" через исключение
                i = 0
    except StopNow:
        return "остановили"

g = controlled_counter()
print(next(g), next(g), next(g))
print("reset via throw(ValueError)")
g.throw(ValueError("reset"))
print(next(g), next(g))

try:
    g.throw(StopNow())
except StopIteration as e:
    print("Завершился, value:", e.value)



## 12. Частые ошибки и как их “почувствовать”



### Ошибка 1: “переиспользовать” генератор

Генератор — одноразовый: если ты его “съел”, он пустой.
Решение: **создавай новый** генератор или собирай в список (если объём нормальный).


In [ ]:

def gen():
    for i in range(3):
        yield i

g = gen()
print(list(g))
print(list(g))  # пусто
print(list(gen()))  # новый генератор -> снова работает



### Ошибка 2: “забыть, что генератор ленивый”

Если внутри генератора есть `print`, ты увидишь его только при итерации.


In [ ]:

def loud():
    print("создали?")
    yield 1
    print("после yield")

g = loud()
print("пока тишина")
print(next(g))



### Ошибка 3: замыкания и “последнее значение” в генераторных выражениях

Похожая ловушка встречается в лямбдах в цикле.
С генераторами тоже можно неожиданно “поймать” переменную поздно.

Сравни:


In [ ]:

funcs = []
for i in range(3):
    funcs.append(lambda: i)

print([f() for f in funcs])  # все 2

# фикс: захватить значение через аргумент по умолчанию
funcs2 = []
for i in range(3):
    funcs2.append(lambda i=i: i)

print([f() for f in funcs2])  # 0,1,2



## 13. “Мини-проект”: свой `itertools` руками

Сделаем несколько полезных генераторов:
- `chunked(iterable, size)` — выдаёт куски списка/потока
- `flatten(iterable_of_iterables)` — расплющивает
- `unique(iterable)` — убирает повторы, сохраняя порядок


In [ ]:

def chunked(iterable, size):
    if size <= 0:
        raise ValueError("size должен быть > 0")
    buf = []
    for x in iterable:
        buf.append(x)
        if len(buf) == size:
            yield buf
            buf = []
    if buf:
        yield buf

def flatten(iterable_of_iterables):
    for it in iterable_of_iterables:
        yield from it

def unique(iterable):
    seen = set()
    for x in iterable:
        if x not in seen:
            seen.add(x)
            yield x

print(list(chunked(range(10), 3)))
print(list(flatten([[1,2], [], [3], [4,5]])))
print(list(unique([1,2,1,3,2,4,4,5])))



## 14. Упражнения (сразу с проверкой)

Сделай упражнения, изменяя код в ячейках. Подсказки будут в комментариях.



### Упражнение 1
Напиши генератор `fibs()` — бесконечный генератор чисел Фибоначчи:  
$0,1,1,2,3,5,8,\dots$

А затем возьми первые 15 через `take()`.


In [ ]:

def fibs():
    # TODO: реализуй
    a, b = 0, 1
    while True:
        yield a
        a, b = b, a + b

print(list(take(fibs(), 15)))



### Упражнение 2
Напиши генератор `grep_lines(path, pattern)`, который:
- читает файл построчно,
- выдаёт только строки, содержащие `pattern`,
- возвращает строки уже без `\n` на конце.

Проверь на `demo_big_text.txt`.


In [ ]:

def grep_lines(path, pattern):
    # TODO: реализуй
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if pattern in line:
                yield line.rstrip("\n")

print(list(grep_lines(p, "ir")))



### Упражнение 3
Сделай генератор `running_max(iterable)`, который выдаёт текущий максимум на каждом шаге.

Пример:
- вход: `2, 1, 5, 3`
- выход: `2, 2, 5, 5`


In [ ]:

def running_max(iterable):
    it = iter(iterable)
    current = next(it)
    yield current
    for x in it:
        if x > current:
            current = x
        yield current

print(list(running_max([2, 1, 5, 3, 10, 7])))



## 15. Короткая шпаргалка

- `yield x` — “верни x и поставь функцию на паузу”
- `next(g)` — получить следующий элемент
- `for x in g:` — удобный перебор
- `g.close()` — закрыть генератор (запустит `finally`)
- `g.send(v)` — отправить значение внутрь (`x = yield ...`)
- `g.throw(E)` — бросить исключение внутрь
- `yield from it` — отдать управление другому итератору/генератору

---

Если хочешь, можешь написать, **какие примеры тебе ближе** (парсинг логов, обработка CSV, чтение huge-файлов, обработка фотографий/метаданных, телеграм-бот), и я добавлю в ноутбук “практический блок” именно под твою задачу.
